# 🧊 Apache Iceberg com Apache Spark

## Cenário: E-commerce de Produtos Eletrônicos

### Modelo ER
```
┌──────────────────┐       ┌──────────────────┐
│    clientes      │       │     pedidos      │
│──────────────────│       │──────────────────│
│ id_cliente (PK)  │──────<│ id_pedido (PK)   │
│ nome             │       │ id_cliente (FK)  │
│ email            │       │ produto          │
│ cidade           │       │ quantidade       │
└──────────────────┘       │ valor_total      │
                           │ status           │
                           │ data_pedido      │
                           └──────────────────┘
```

### Diferencial do Iceberg
- **Particionamento oculto** — sem expor ao usuário
- **Schema Evolution** — adicionar/remover colunas sem reescrever dados
- **Time Travel** — snapshots imutáveis
- **Row-level deletes** — muito eficiente

## 1. Configuração do Ambiente com Iceberg

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

# Versão do Iceberg compatível com Spark 3.5
ICEBERG_VERSION = '1.5.0'
SCALA_VERSION   = '2.12'

spark = (
    SparkSession.builder
    .appName('Iceberg - E-commerce')
    .config(
        'spark.jars.packages',
        f'org.apache.iceberg:iceberg-spark-runtime-3.5_{SCALA_VERSION}:{ICEBERG_VERSION}'
    )
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', '/tmp/iceberg/warehouse')
    .config('spark.sql.defaultCatalog', 'local')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
print(f'PySpark versão: {pyspark.__version__}')
print('✅ SparkSession com Apache Iceberg iniciada com sucesso!')

:: loading settings :: url = jar:file:/home/gabrielmaciel/apache_spark_delta_lake_apache_iceberg/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/gabrielmaciel/.ivy2/cache
The jars for the packages stored in: /home/gabrielmaciel/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5f90d7f9-1652-407e-8f81-bafa6c201c34;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.0/iceberg-spark-runtime-3.5_2.12-1.5.0.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0!iceberg-spark-runtime-3.5_2.12.jar (10970ms)
:: resolution report :: resolve 1556ms :: artifacts dl 10983ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| searc

PySpark versão: 3.5.1
✅ SparkSession com Apache Iceberg iniciada com sucesso!


## 2. Criar namespace e tabelas Iceberg

In [2]:
# Criar namespace (equivalente a schema/database)
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.ecommerce')
print('✅ Namespace criado: local.ecommerce')

✅ Namespace criado: local.ecommerce


In [3]:
# Criar tabela clientes como Iceberg
spark.sql('''
    CREATE TABLE IF NOT EXISTS local.ecommerce.clientes (
        id_cliente INT,
        nome       STRING,
        email      STRING,
        cidade     STRING
    ) USING iceberg
''')

# Criar tabela pedidos como Iceberg com particionamento
spark.sql('''
    CREATE TABLE IF NOT EXISTS local.ecommerce.pedidos (
        id_pedido   INT,
        id_cliente  INT,
        produto     STRING,
        quantidade  INT,
        valor_total DOUBLE,
        status      STRING,
        data_pedido STRING
    ) USING iceberg
    PARTITIONED BY (status)
''')

print('✅ Tabelas Iceberg criadas com particionamento por status!')

✅ Tabelas Iceberg criadas com particionamento por status!


## 3. INSERT — Inserindo dados

In [4]:
# INSERT INTO clientes
spark.sql('''
    INSERT INTO local.ecommerce.clientes VALUES
    (1, 'Ana Souza',    'ana@email.com',    'São Paulo'),
    (2, 'Bruno Lima',   'bruno@email.com',  'Rio de Janeiro'),
    (3, 'Carla Melo',   'carla@email.com',  'Curitiba'),
    (4, 'Diego Faria',  'diego@email.com',  'Belo Horizonte')
''')

print('✅ INSERT em clientes executado')
spark.sql('SELECT * FROM local.ecommerce.clientes').show()

✅ INSERT em clientes executado


+----------+-----------+---------------+--------------+
|id_cliente|       nome|          email|        cidade|
+----------+-----------+---------------+--------------+
|         1|  Ana Souza|  ana@email.com|     São Paulo|
|         2| Bruno Lima|bruno@email.com|Rio de Janeiro|
|         3| Carla Melo|carla@email.com|      Curitiba|
|         4|Diego Faria|diego@email.com|Belo Horizonte|
+----------+-----------+---------------+--------------+



In [5]:
# INSERT INTO pedidos
spark.sql('''
    INSERT INTO local.ecommerce.pedidos VALUES
    (101, 1, 'Notebook Dell',   1, 4500.00, 'aprovado',  '2024-01-10'),
    (102, 2, 'iPhone 15',       1, 5800.00, 'aprovado',  '2024-01-11'),
    (103, 3, 'Smart TV 55',     1, 2900.00, 'pendente',  '2024-01-12'),
    (104, 4, 'Fone Bluetooth',  2,  350.00, 'aprovado',  '2024-01-13'),
    (105, 1, 'SSD 1TB',         1,  480.00, 'pendente',  '2024-01-14')
''')

print('✅ INSERT em pedidos executado')
spark.sql('SELECT * FROM local.ecommerce.pedidos ORDER BY id_pedido').show()

✅ INSERT em pedidos executado
+---------+----------+--------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|       produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+--------------+----------+-----------+--------+-----------+
|      101|         1| Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      102|         2|     iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      103|         3|   Smart TV 55|         1|     2900.0|pendente| 2024-01-12|
|      104|         4|Fone Bluetooth|         2|      350.0|aprovado| 2024-01-13|
|      105|         1|       SSD 1TB|         1|      480.0|pendente| 2024-01-14|
+---------+----------+--------------+----------+-----------+--------+-----------+



## 4. UPDATE — Atualizando registros no Iceberg

In [6]:
# UPDATE: aprovar pedidos pendentes
spark.sql('''
    UPDATE local.ecommerce.pedidos
    SET status = 'aprovado'
    WHERE status = 'pendente'
''')

print('✅ UPDATE executado — pedidos pendentes aprovados')
spark.sql('SELECT * FROM local.ecommerce.pedidos ORDER BY id_pedido').show()

✅ UPDATE executado — pedidos pendentes aprovados
+---------+----------+--------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|       produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+--------------+----------+-----------+--------+-----------+
|      101|         1| Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      102|         2|     iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      103|         3|   Smart TV 55|         1|     2900.0|aprovado| 2024-01-12|
|      104|         4|Fone Bluetooth|         2|      350.0|aprovado| 2024-01-13|
|      105|         1|       SSD 1TB|         1|      480.0|aprovado| 2024-01-14|
+---------+----------+--------------+----------+-----------+--------+-----------+



## 5. DELETE — Removendo registros

In [7]:
# DELETE: remover pedidos de baixo valor
spark.sql('''
    DELETE FROM local.ecommerce.pedidos
    WHERE valor_total < 400
''')

print('✅ DELETE executado — pedidos com valor < R$400 removidos')
spark.sql('SELECT * FROM local.ecommerce.pedidos ORDER BY id_pedido').show()

✅ DELETE executado — pedidos com valor < R$400 removidos
+---------+----------+-------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|      produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+-------------+----------+-----------+--------+-----------+
|      101|         1|Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|      102|         2|    iPhone 15|         1|     5800.0|aprovado| 2024-01-11|
|      103|         3|  Smart TV 55|         1|     2900.0|aprovado| 2024-01-12|
|      105|         1|      SSD 1TB|         1|      480.0|aprovado| 2024-01-14|
+---------+----------+-------------+----------+-----------+--------+-----------+



## 6. MERGE (UPSERT) no Iceberg

In [8]:
# Criar tabela temporária com novos dados
novos_pedidos = [
    (102, 2, 'iPhone 15',   1, 5900.00, 'cancelado', '2024-01-11'),  # UPDATE
    (106, 3, 'Tablet iPad', 1, 3200.00, 'aprovado',  '2024-01-15'),  # INSERT
]
df_novos = spark.createDataFrame(
    novos_pedidos,
    ['id_pedido', 'id_cliente', 'produto', 'quantidade', 'valor_total', 'status', 'data_pedido']
)
df_novos.createOrReplaceTempView('novos_pedidos')

spark.sql('''
    MERGE INTO local.ecommerce.pedidos AS destino
    USING novos_pedidos AS origem
    ON destino.id_pedido = origem.id_pedido
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''')

print('✅ MERGE executado!')
spark.sql('SELECT * FROM local.ecommerce.pedidos ORDER BY id_pedido').show()

✅ MERGE executado!
+---------+----------+-------------+----------+-----------+---------+-----------+
|id_pedido|id_cliente|      produto|quantidade|valor_total|   status|data_pedido|
+---------+----------+-------------+----------+-----------+---------+-----------+
|      101|         1|Notebook Dell|         1|     4500.0| aprovado| 2024-01-10|
|      102|         2|    iPhone 15|         1|     5900.0|cancelado| 2024-01-11|
|      103|         3|  Smart TV 55|         1|     2900.0| aprovado| 2024-01-12|
|      105|         1|      SSD 1TB|         1|      480.0| aprovado| 2024-01-14|
|      106|         3|  Tablet iPad|         1|     3200.0| aprovado| 2024-01-15|
+---------+----------+-------------+----------+-----------+---------+-----------+



## 7. Time Travel — Snapshots do Iceberg

In [10]:
# Ver histórico de snapshots
snapshots_df = spark.sql('SELECT snapshot_id, committed_at, operation FROM local.ecommerce.pedidos.snapshots')
snapshots_df.show(truncate=False)

# Pegar o snapshot_id mais antigo automaticamente
primeiro_snapshot = snapshots_df.orderBy('committed_at').first()['snapshot_id']

print(f'\n📸 Estado inicial (snapshot_id={primeiro_snapshot}):')
spark.sql(f'''
    SELECT * FROM local.ecommerce.pedidos VERSION AS OF {primeiro_snapshot}
''').show()

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|8723956584378062988|2026-04-27 16:57:57.151|append   |
|158650469088951976 |2026-04-27 16:58:00.356|overwrite|
|2283856248426815530|2026-04-27 16:58:02.443|overwrite|
|5961036190945373961|2026-04-27 16:58:12.608|overwrite|
+-------------------+-----------------------+---------+


📸 Estado inicial (snapshot_id=8723956584378062988):
+---------+----------+--------------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|       produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+--------------+----------+-----------+--------+-----------+
|      103|         3|   Smart TV 55|         1|     2900.0|pendente| 2024-01-12|
|      105|         1|       SSD 1TB|         1|      480.0|pendente| 2024-01-14|
|      101|         1| Notebook Dell|         1|     4500.0|aprovado| 2024-01-10|
|     

## 8. Schema Evolution — Adicionar coluna sem reescrever dados

In [11]:
# Adicionar coluna 'avaliacao' sem impactar dados existentes
spark.sql('''
    ALTER TABLE local.ecommerce.pedidos
    ADD COLUMN avaliacao INT
''')

print('✅ Coluna avaliacao adicionada (Schema Evolution)')
spark.sql('SELECT * FROM local.ecommerce.pedidos ORDER BY id_pedido').show()

✅ Coluna avaliacao adicionada (Schema Evolution)
+---------+----------+-------------+----------+-----------+---------+-----------+---------+
|id_pedido|id_cliente|      produto|quantidade|valor_total|   status|data_pedido|avaliacao|
+---------+----------+-------------+----------+-----------+---------+-----------+---------+
|      101|         1|Notebook Dell|         1|     4500.0| aprovado| 2024-01-10|     NULL|
|      102|         2|    iPhone 15|         1|     5900.0|cancelado| 2024-01-11|     NULL|
|      103|         3|  Smart TV 55|         1|     2900.0| aprovado| 2024-01-12|     NULL|
|      105|         1|      SSD 1TB|         1|      480.0| aprovado| 2024-01-14|     NULL|
|      106|         3|  Tablet iPad|         1|     3200.0| aprovado| 2024-01-15|     NULL|
+---------+----------+-------------+----------+-----------+---------+-----------+---------+

